# Figure 4.16: Designing low-pass filters

*The Discrete Algebra of the Fourier Transform*, Gabriel Peyré.

Low-pass filters with increasingly wide raised-cosine transitions. The columns show the centered impulse response, the prescribed DFT samples, and the densely sampled FIR frequency response.

[Read this figure in the book](https://mathematical-tours.github.io/daft-sources/DAFT-EN.pdf#page=100).

## Prepare the example

Run the cells in order in Jupyter or Google Colab. The first cell installs missing Python packages and downloads the figure sources and input data once. Subsequent runs use the cached download. Python 3.10 or later is required.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import io
import os
import subprocess
import sys
import urllib.request
import zipfile

packages = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
            "PIL": "Pillow", "sympy": "sympy"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

archive_url = os.environ.get("DAFT_FIGURE_ARCHIVE", 'https://mathematical-tours.github.io/daft-sources/downloads/daft-figures-2e843bbe89f0.zip')
archive_sha = '2e843bbe89f0e274849b6896eb16bd9f7686521a8ca98f633240bfc06ea2d5b1'
workspace = Path(os.environ.get("DAFT_FIGURE_WORKDIR", "daft-figure-files")) / archive_sha[:12]
cache = workspace / "sources.zip"
workspace.mkdir(parents=True, exist_ok=True)
data = cache.read_bytes() if cache.exists() else urllib.request.urlopen(archive_url, timeout=60).read()
if hashlib.sha256(data).hexdigest() != archive_sha:
    raise ValueError("The figure-source download is incomplete or has changed. Download it again.")
cache.write_bytes(data)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    archive.extractall(workspace)
ROOT = (workspace / "daft-figures").resolve()
__file__ = str(ROOT / 'figures-code/chapter-fourier-applications/low-pass-filters/generate.py')
print("Figure sources ready.")


## Generate the figure

This is the plotting code used by the book. Change its parameters and rerun this cell, then the display cell below, to explore the example.

In [ ]:
from pathlib import Path
import sys
ROOT=Path(__file__).resolve().parents[3]
sys.path.insert(0,str(ROOT/"tools"))
sys.path.insert(0,str(ROOT/"listings"))
from figure_style import *
from image_processing import smooth_lowpass_kernel,filter_response
n=65;k=np.arange(-(n//2),n//2+1);frequency=np.fft.fftshift(np.fft.fftfreq(n))
fig,axes=plt.subplots(3,3,figsize=(9,6.3))
for row,epsilon in enumerate([0,.4,1]):
    g=smooth_lowpass_kernel(n,epsilon);H=np.fft.fftshift(np.fft.fft(g)).real
    discrete(axes[row,0],k,np.fft.fftshift(g));axes[row,0].set_ylabel(rf'$\epsilon={epsilon:g}$')
    axes[row,1].plot(2*frequency,H,'o',ms=2.5,color=TEAL)
    omega,curve=filter_response(g,2048);assert np.max(abs(curve.imag))<1e-13
    axes[row,2].plot(omega/np.pi,curve.real,color=TEAL)
    for col,ax in enumerate(axes[row]):axes_box(ax,'kernel index' if col==0 else r'$\omega/\pi$')
    axes[row,0].set(xlim=(-20,20),ylim=(-.14,.56));axes[row,1].set(xlim=(-1,1),ylim=(-.15,1.15));axes[row,2].set(xlim=(-1,1),ylim=(-.15,1.15))
for ax,title in zip(axes[0],['Impulse response','Prescribed samples','FIR frequency response']):ax.set_title(title)
save(fig,'chapter-fourier-applications','low-pass-filters')


## Display and save

In [ ]:
from IPython.display import Image, FileLink, display

display(Image(filename=str(ROOT / 'build/figure-previews/chapter-fourier-applications/low-pass-filters.png')))
display(FileLink(os.path.relpath(ROOT / 'figures/chapter-fourier-applications/low-pass-filters/figure.pdf', Path.cwd()),
                 result_html_prefix="Download the vector PDF: "))
